<a href="https://colab.research.google.com/github/zakateryila/Zaka-Teryila-Peter-/blob/main/Day_5_Lab_Design_an_AI_System_on_Paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## AI System Design Document

### 1. System Overview & Problem Statement
* **Product Name:** Clinic Triage Bot (WhatsApp-based)
* **The Core Problem:** Patients need a fast, accessible way to find the right clinical department and schedule appointments based on symptoms. However, medical cases can escalate quickly. The system must accurately triage symptoms, book appointments, and instantly detect emergencies to route them to human clinical staff.
* **What we are building:** An automated, conversational WhatsApp interface powered by an orchestrator that evaluates patient symptom descriptions, matches them to clinic specialties, schedules appointments via integration, and enforces strict guardrails to flag high-risk situations for immediate human escalation.

### 2. The Five Layers Architecture

| Layer | Component details | Key Decisions | Explanations |
| :--- | :--- | :--- | :--- |
| **Interface** | WhatsApp Business API, Twilio Gateway, Custom webhook handler. | Text/voice-based WhatsApp conversational UI. | Maximizes accessibility for patients across demographics without requiring them to download a dedicated mobile app. |
| **Orchestration** | Python FastAPI Backend, Guardrails SDK, Celery for async scheduling, Twilio API integration. | Custom router with deterministic fallback instead of pure LLM agents. | Medical triage requires predictable, safety-first paths. The orchestrator enforces strict validation and immediately intercepts emergency keywords before calling LLMs. |
| **Model** | **Primary:** Gemini 2.5 Flash (for speed and JSON schema compliance).<br>**Secondary:** Local BioBERT classifier (backup triage fallback). | **Model Choice & Costs:**<br>- **Model:** Gemini 2.5 Flash.<br>- **Why over alternative (Gemini 2.5 Pro):** We need extremely low latency (< 1.5 seconds) for interactive chat. Pro is too slow and overly expensive for simple structured routing classification.<br>- **Est. Cost:** ~$0.000075 per turn (Input: $0.075/1M tokens, Output: $0.30/1M tokens). Average response time: **~0.8s to 1.2s**. | Does this need a model? Yes, because patients describe symptoms in conversational, non-standard natural language. Standard keyword lookup cannot reliably generalize to complex descriptive symptoms. |
| **Data** | **Relational:** PostgreSQL (Appointments, Patient metadata).<br>**Cache:** Redis (Session history, Rate-limiting).<br>**Vector Store:** pgvector (Clinical guideline documents / FAQs). | Postgres + pgvector combined. | Reduces operational complexity by keeping patient relational records and retrieval-augmented guidelines in a single, ACID-compliant database. |
| **Operations** | Langfuse (tracing), Prometheus & Grafana (latency & rate metrics), Human-in-the-loop (HITL) clinical escalation dashboard. | Multi-tier alerting and active human nurse takeover queue. | Critical for patient safety. Allows operational staff or triage nurses to seamlessly take over the WhatsApp session if the system detects confusion, anxiety, or borderline clinical signs. |

### 3. Design Decisions Table

| Decision | What I Chose | Why | What I Rejected |
| :--- | :--- | :--- | :--- |
| **Interface** | **WhatsApp Business API via Twilio Gateway** | Maximizes accessibility across user demographics. No app installs required; lowers friction for patients who need quick triage routing. | **Custom Mobile Web App**<br>Rejected due to high friction; sick patients are highly unlikely to download an app or log in during a triage scenario. |
| **Model for Main Task** | **Gemini 2.5 Flash** with JSON Structured Outputs | Low latency (~0.8s - 1.2s), extremely low cost ($0.075/1M input tokens), and robust schema compliance for reliable structured triage responses. | **Gemini 2.5 Pro or Fine-tuned Clinical LLM**<br>Rejected because Pro has much higher latency/cost, and fine-tuning a self-hosted clinical LLM introduces immense infrastructure overhead without performance benefits for classification routing. |
| **Where Data is Stored** | **PostgreSQL (with pgvector extension)** | Consolidates transactional database logs, patient scheduling, and vector embeddings of medical guidelines into a single ACID-compliant database. | **Dedicated Vector DB (e.g., Pinecone/Milvus) + Postgres**<br>Rejected due to needless complexity. Our database is small (<1,000 guidelines), and maintaining two separate systems increases sync latency and cost. |
| **Human in the Loop** | **Active clinical nurse queue takeover dashboard** | Ensures patient safety. If the model returns low confidence (<0.85) or flags high risk, an on-call nurse seamlessly takes over the conversation. | **Fully Autonomous Triage & Scheduling**<br>Rejected because edge-case medical symptoms or borderline emergencies can be misrouted, presenting severe clinical safety risks and legal liabilities. |

### 4. Failure Notes & Fallbacks

1. **Failure Mode 1: LLM Under-triage of Critical Symptoms (Silent Failure)**
   * **What happens:** A patient describes symptoms of an acute cardiac event using vague terms like "indigestion combined with arm tingling". The LLM hallucinates/under-triages this as low-risk stomach acidity.
   * **Concrete Fallback:** Implement a deterministic pre-processing step using local regex/keyword-matching scanner (sensitive to high-risk keywords like "chest", "numbness", "breath"). If triggered, bypass LLM completely, automatically send a template warning instructing the patient to call emergency services directly, and alert the on-call medical operator via the nurse takeover dashboard.

2. **Failure Mode 2: Model API is Down or Rate-Limited at 9am Monday Peak**
   * **What happens:** High traffic causes HTTP 429 Rate Limits or complete API service downtime, leaving patient chats hanging.
   * **Concrete Fallback:** Implement an immediate local fallback classifier (e.g. BioBERT run locally on CPU). If the Gemini API call fails or times out after 2 seconds, retry once. If still failed, run the query through local BioBERT. If local classification fails, immediately push the patient to the human clinical queue and send a WhatsApp response: *"We are experiencing higher volume than usual. A clinical triage coordinator will review your symptoms shortly. If this is an emergency, please dial 911."*

3. **Failure Mode 3: User sends absurd, hostile, or unhandled language inputs**
   * **What happens:** Patient inputs foreign languages, spam, or hostile content designed to jailbreak the prompt or bypass the booking logic.
   * **Concrete Fallback:** Apply a language detection filter (e.g., fastText) at the orchestration layer. If any language other than English/Spanish is detected, or if toxicity classification exceeds 0.8, immediately route the conversation to the human triage queue and return a standardized multi-language message: *"Please hold while we connect you with an agent who can assist you in your preferred language."*

### 5. Data Privacy, Retention & PII (Step 6)

* **What is stored:** We store Patient Phone Numbers (for WhatsApp communication routing), Medical Record Numbers (MRN), Booking Date/Time, and Triage Classification logs (specialty assigned & urgency level).
* **Retention Period:**
  * *Relational Database (PostgreSQL):* Retained permanently under secure electronic health record (EHR) guidelines.
  * *In-Memory Cache (Redis Session):* Automatically deleted via TTL (Time-to-live) set to 30 minutes after last interaction.
* **PII/PHI Security & Visibility:**
  * Access to the database is restricted strictly to licensed clinical triage staff via secure VPN and OAuth2/MFA clinical dashboard.
  * **Data Stripping before Model:** Prior to sending the text context to the Gemini API, our Orchestrator runs a local PII/PHI scrubber (e.g., Presidio). It strips names, dates of birth, addresses, and phone numbers from the symptom text. Only anonymous text *("Patient describes a dull pressure in the left side of... ")* is transmitted to the external model API to maintain compliance.

In [ ]:
### 6. Refined Excalidraw Blueprint Map (with Closed-Loop Feedback)

Use this final map to lay out your diagram in [Excalidraw](https://excalidraw.com/). Flow top-to-bottom:

1. 🔵 **[Interface] Patient WhatsApp Widget**
   * ⬇️ *Arrow: "Raw symptoms (text/audio) + Patient ID"* ⬇️
2. 🔵 **[Orchestration] FastAPI Backend Backend** (clean text, strip PII via Presidio, check length)
   * ⬇️ *Arrow: "Cleaned, anonymized text"* ⬇️
   * ↔️ *Arrow: "Lookup matches"* ↔️ 🟣 **[Data] PostgreSQL (Clinical Guidelines & Historical Few-Shots)**
3. 🔵 **[Orchestration] Prompt Assembler**
   * ⬇️ *Arrow: "Assembled Prompt (System instructions + Guidelines + 3 Few-Shot Examples)"* ⬇️
4. 🟢 **[Model] Gemini 2.5 Flash Call**
   * ⬇️ *Arrow: "Structured JSON {specialty, urgency, reason, confidence}"* ⬇️
   * 🔴 *Failure Arrow (Malformed JSON): "Retry once -> then default route to general triage queue"* 🔴
5. 🟠 **[Orchestration] Router (Decision Diamond: Is Confidence ≥ 0.85?)**
   * ⬇️ *Arrow (Yes): "Direct booking pipeline"* ⬇️
     * 🔵 **[Orchestration] Auto-Scheduler** ➡️ 🟣 **[Data] Postgres Appointment DB**
   * ⬇️ *Arrow (No): "Draft + low-confidence flag"* ⬇️
     * 🔵 **[Operations/Interface] Clinical Nurse Takeover Dashboard**
6. 🔵 **[Operations] Feedback Harvester**
   * ⬅️ *Arrow: "Nurse-corrected/edited triage labels & feedback loop"* ⬅️ **[Operations/Interface] Clinical Nurse Takeover Dashboard**
   * ⬇️ *Dashed Arrow: "Save corrected examples asynchronously"* ⬇️
   * 🟣 **[Data] PostgreSQL (Historical Few-Shots Database)** *(This closes the loop: corrected triage cases become tomorrow's few-shot context!)*